Generate a chart for a validation song and compare it with the ground truth

Runs top to bottom: setup, load validation data, load model, generate, export.

In [1]:
import sys, json, warnings
import torch
from pathlib import Path
from hydra import initialize_config_dir, compose

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_DIR = PROJECT_ROOT / "conf"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with initialize_config_dir(version_base=None, config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="train")

experiment_config = cfg.experiment
hyperparams = cfg.hyperparams
segment_frames = hyperparams["segment_frames"]
batch_size = hyperparams["batch_size"]

warnings.filterwarnings("ignore", message=".*nested tensors.*")

In [ ]:
from tracers.datasets.dataset import GlobalAudioCollator
from tracers.datasets import AutoOsu4KDataset, split_song_ids, make_dataloader

ROOT = PROJECT_ROOT / "dataset"
AUDIO_DIR = ROOT / "audio"
BEATMAP_DIR = ROOT / "beatmap" / "4keys"

# The base dataset is only used to reproduce the same song split as training.
base_dataset = AutoOsu4KDataset(
    beatmap_dir=BEATMAP_DIR,
    audio_dir=AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="random",
    audio_cache_size=8,
)
_, val_ids, _ = split_song_ids(
    base_dataset.song_ids,
    val_fraction=0.10,
    test_fraction=0.10,
    seed=42,
)

val_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="all",
    song_ids=val_ids,
)
val_loader = make_dataloader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=GlobalAudioCollator(
        AUDIO_DIR,
        song_ids=val_ids,
        cache_size=2
    )
)



In [3]:
from tracers.models.transformer import TracerGlobalNpModel
from tracers.training.trainer import Trainer
from tracers.generate.converter import predictions_to_json, export_generation_debug

model = TracerGlobalNpModel(hyperparams).to(device)

# The optimizer is only here because load_checkpoint() also restores its state.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hyperparams["learning_rate"],
    weight_decay=hyperparams["weight_decay"],
)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_loader=None,
    valid_loader=None,
    device=device,
    experiment_config=experiment_config,
    hyperparams=hyperparams,
)

CHECKPOINT = PROJECT_ROOT / "notebooks/checkpoints/best.pt"
trainer.load_checkpoint(CHECKPOINT)

Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\best.pt (epoch 39, best val loss 0.1490)


In [ ]:
context_frames = 500

val_ar_dataset = AutoOsu4KDataset(
    beatmap_dir=BEATMAP_DIR,
    audio_dir=AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="all",
    stride_frames=segment_frames - context_frames,
    song_ids=val_ids,
    audio_cache_size=8,
)

ar_collator = None

if hasattr(trainer.model, "prepare_global_context"):
    ar_collator = GlobalAudioCollator(
        AUDIO_DIR,
        song_ids=val_ids,
        cache_size=2,
    )

val_ar_loader = make_dataloader(
    val_ar_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    collate_fn=ar_collator,
)

In [4]:
# ---------- settings ----------
CHART_ID = None          # e.g. "1055661-148-4"; None = first chart in the validation set
BPM = 120                # only used to place notes in the exported JSON
OFFSET_MS = 0
LANE_MAP = (0, 2, 4, 6)

TEMPERATURE = None        # None = greedy
TOP_K = None
TOP_P = None
SEED = 0
SAVE_DEBUG_CSV = True

# ---------- collect the chart's segments ----------
INPUT_KEYS = ["specs", "beat_phase", "beat_num", "difficulty", "valid_frames"]
segments, found = [], False
full_song_specs = None

for batch in val_loader:
    for i, cid in enumerate(batch["chart_id"]):
        if CHART_ID is None:
            CHART_ID = cid

        if cid != CHART_ID:
            if found:
                break
            continue

        found = True

        # Find the full-song spectrogram associated with this sample.
        global_idx = int(batch["global_indices"][i].item())
        song_specs = batch["global_specs"][global_idx]

        if full_song_specs is None:
            full_song_specs = song_specs

        segments.append({
            "start": int(batch["start_frame"][i]),
            "sample": {k: batch[k][i:i+1] for k in INPUT_KEYS},
            "actual": batch["actions_target"][i].cpu(),
        })
    else:
        continue
    break

segments.sort(key=lambda s: s["start"])

print(f"Chart {CHART_ID}: {len(segments)} segments")
print("Full song specs:", full_song_specs.shape if full_song_specs is not None else None )

Chart 1055661-148-4: 8 segments
Full song specs: torch.Size([3, 14946, 80])


In [5]:
model = trainer.model
device = next(model.parameters()).device


full_global_context = None

if hasattr(model, "prepare_global_context"):
    model.eval()

    with torch.no_grad():
        full_global_context = model.prepare_global_context(
            global_specs=[full_song_specs],
            global_indices=torch.tensor([0]),
            start_frames=torch.tensor([0]),
            seq_len=full_song_specs.size(1) if full_song_specs is not None else None,
            device=device,
            dtype=full_song_specs.dtype if full_song_specs is not None else None,
        )

In [6]:
torch.manual_seed(SEED)

generated_segments, actual_segments, debug_segments = [], [], []
seed_action = 0

for seg in segments:
    sample = seg["sample"]

    generate_kwargs = {}

    if full_global_context is not None:
        start = seg["start"]
        seq_len = sample["specs"].size(2)

        global_context = full_global_context[:, start:start + seq_len, :]

        # Handle a potentially padded final segment.
        if global_context.size(1) < seq_len:
            global_context = torch.nn.functional.pad(
                global_context,
                (0, 0, 0, seq_len - global_context.size(1))
            )

        generate_kwargs["global_context"] = global_context

    results = trainer.generate(
        sample,
        seed_action=seed_action,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        top_p=TOP_P,
        **generate_kwargs,
    )

    valid = results["valid_frames"][0]
    n = int(valid.sum().item())

    predictions = results["action_predictions"][0][:n]
    actual = seg["actual"][:n]

    generated_segments.append(predictions)
    actual_segments.append(actual)

    debug_segments.append({
        "start": seg["start"],
        "action_inputs": results["action_inputs"][0][:n],
        "action_predictions": predictions,
        "np_pred": results["np_pred"][0][:n],
        "ns_logits": results["ns_logits"][0][:n],
        "actual": actual,
        "valid_frames": valid[:n],
    })

    seed_action = int(predictions[-1].item())

    print(
        f"Frame {seg['start']:6d} | frames {n} | "
        f"active {(predictions != 0).sum().item():4d} "
        f"(actual {(actual != 0).sum().item():4d}) | "
        f"next seed {seed_action}"
    )

generated = torch.cat(generated_segments)
actual = torch.cat(actual_segments)

print(f"\nDone. {len(generated)} frames ({len(generated) * 0.01:.1f} s)")
print(
    f"Active frames: generated {(generated != 0).sum().item()} "
    f"| actual {(actual != 0).sum().item()}"
)

Frame      0 | frames 2000 | active   20 (actual   68) | next seed 0
Frame   2000 | frames 2000 | active   50 (actual   56) | next seed 0
Frame   4000 | frames 2000 | active   86 (actual   50) | next seed 0
Frame   6000 | frames 2000 | active   57 (actual   50) | next seed 0
Frame   8000 | frames 2000 | active    1 (actual   64) | next seed 0
Frame  10000 | frames 2000 | active   43 (actual   63) | next seed 0
Frame  12000 | frames 2000 | active   43 (actual   57) | next seed 0
Frame  14000 | frames 946 | active    5 (actual   12) | next seed 0

Done. 14946 frames (149.5 s)
Active frames: generated 305 | actual 420


In [7]:
first_start = segments[0]["start"]
valid_all = torch.ones(len(generated), dtype=torch.bool)

def to_events(actions):
    return predictions_to_json(
        action_predictions=actions,
        bpm=BPM,
        start_frame=first_start,
        offset_ms=OFFSET_MS,
        lane_map=LANE_MAP,
        valid_frames=valid_all,
    )

output_dir = Path(f"charts/{CHART_ID}")
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "generated_np_global_chart.json", "w", encoding="utf-8") as f:
    json.dump(to_events(generated), f, indent=4)

with open(output_dir / "actual_full_chart.json", "w", encoding="utf-8") as f:
    json.dump(to_events(actual), f, indent=4)

print("Saved generated_np_global_chart.json and actual_full_chart.json")

if SAVE_DEBUG_CSV:
    export_generation_debug(debug_segments, output_path=str(output_dir / "generation_np_global_debug.csv"), top_k=5)

Saved generated_np_global_chart.json and actual_full_chart.json
Exported 14946 frame records to: charts\1055661-148-4\generation_np_global_debug.csv
